# QuantStream — Run 005: đối chứng inference với trọng số cố định

**Bản sửa runtime Retry001: Run All một lần, không có MODE và không fine-tune lại.** Runner dùng adapter seed 42 đã chạy ở Run 004, nạp dạng PEFT không merge và so sánh `eval()` với dropout bật dưới `torch.inference_mode()`.

Bản này sửa bootstrap sau lỗi setup NumPy, dùng thư mục/ZIP mới để giữ nguyên evidence lần lỗi. Giữ nguyên checkpoint, adapter, dữ liệu, tám lượt đối chứng và 0 optimizer steps.

Trước khi chạy:

1. Import notebook này vào Kaggle, bật Internet và chọn **Tesla T4 như Run 004**. Bắt đầu session mới.
2. Add Input **Dataset snapshot cũ** đang dùng cho Run 004.
3. Add Input thêm **`QuantStream_Run004_Output.zip`** đã gửi để review. Nếu Kaggle giải nén ZIP thành các thư mục `seed-42`, `seed-123`, `seed-2026`, runner vẫn kiểm tra đúng file/hashes. Không dùng ZIP preflight.
4. Chọn **Run All**. Tải `QuantStream_Run005_Retry001_Output.zip` gửi lại, kể cả khi có lỗi. Nếu lỗi trước khi ZIP được tạo, gửi traceback.

Ngân sách cố định: một lượt zero-shot để đối chiếu và tám lượt cùng adapter, mỗi lượt 512 cửa sổ development đã xem; **0 optimizer steps**. Chưa có kết quả Run 005 trên Kaggle khi phát hành notebook này. Mục tiêu là đo ảnh hưởng của mode/RNG, không tìm model hoặc seed mới.

## 1. Nạp source đã pin

Mỗi source SHA dùng checkout riêng. Dữ liệu và output nằm ngoài thư mục source để giữ Git tree sạch.

In [ ]:
from pathlib import Path
import sys, subprocess, json, zipfile, traceback, importlib.metadata, platform
from IPython.display import display, FileLink

loaded=[name for name in ('peft','torchao','chronos','transformers','quantstream')
        if any(module==name or module.startswith(name+'.') for module in sys.modules)]
if loaded:
    raise RuntimeError('Restart Session rồi Run All: '+', '.join(loaded))
SOURCE_COMMIT='d6bbe46d8452422f9a606704041af3e2ceb8fee8'
WORK=Path('/kaggle/working/quantstream_run005_retry001')
WORK.mkdir(parents=True,exist_ok=True)
CODE=WORK/('source-'+SOURCE_COMMIT[:12])
OUTPUT=WORK/'run-005-inference'
ARCHIVE=WORK.parent/'QuantStream_Run005_Retry001_Output.zip'
if OUTPUT.exists() or ARCHIVE.exists():
    raise FileExistsError('Đã có evidence Run 005; tải lại để review, không xóa nhằm chạy lại.')
if not CODE.exists():
    subprocess.check_call(['git','init','-q',str(CODE)])
    subprocess.check_call(['git','-C',str(CODE),'remote','add','origin',
                           'https://github.com/taiiswibu/quantstream-crypto-research.git'])
    subprocess.check_call(['git','-C',str(CODE),'fetch','-q','--depth','1','origin',SOURCE_COMMIT])
    subprocess.check_call(['git','-C',str(CODE),'checkout','-q','--detach','FETCH_HEAD'])
actual=subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()
if actual!=SOURCE_COMMIT or subprocess.check_output(['git','-C',str(CODE),'status','--porcelain'],text=True).strip():
    raise ValueError('Cần source SHA đã pin và checkout sạch')
sys.path.insert(0,str(CODE))
print('Pinned source:',SOURCE_COMMIT)

## 2. Khôi phục bootstrap đã đối chiếu với probe

Probe Kaggle đã xác minh NumPy **thực sự import là 2.1.3**. Metadata 1.26.4 ở thư mục hệ thống đứng sau là giá trị mà lock cũ ghi; không dùng nó làm pip constraint. Giữ nguyên NumPy, PyTorch/CUDA và toàn bộ package nền.

Cell chỉ gỡ optional `torchao==0.10.0` và cài phần còn thiếu `chronos-forecasting==2.3.2` bằng `--no-deps`. Trước/sau đều so full inventory với lock bằng cùng cách thu thập cũ, đồng thời kiểm tra phiên bản/module path thực, dependencies của stack research và một forward PEFT tổng hợp trên CPU. Không có backward, fit hoặc tải checkpoint trong setup.

Nếu xuất hiện drift ngoài hai thay đổi đã review, dừng và giữ ZIP; không tự sửa lock. Run 004 chưa ghi loaded NumPy version/path, nên metadata inventory khớp không tự chứng minh lịch sử dùng đúng module version này.

In [ ]:
import os, shutil
from quantstream.inference_setup import prepare_runtime
os.environ['HF_HUB_DISABLE_XET']='1'
old_lock=json.loads((CODE/'reports/experiments/run-004-seed-stability/kaggle-preflight-lock-rerun-001.json').read_text())
SETUP=WORK/'setup-evidence'
try:
    setup_report=prepare_runtime(old_lock['environment'],SETUP)
except BaseException:
    OUTPUT.mkdir(parents=True,exist_ok=False)
    if SETUP.exists():
        shutil.copytree(SETUP,OUTPUT/'setup',dirs_exist_ok=True)
    (OUTPUT/'setup-error.txt').write_text(traceback.format_exc(),encoding='utf-8')
    (OUTPUT/'status.json').write_text(json.dumps({'status':'setup_failed','optimizer_steps':0}),encoding='utf-8')
    with zipfile.ZipFile(ARCHIVE,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in OUTPUT.rglob('*'):
            if path.is_file():
                archive.write(path,path.relative_to(OUTPUT))
    display(FileLink(str(ARCHIVE)))
    raise
print('Setup completed. NumPy imported:',setup_report['loaded_runtime']['modules']['numpy']['version'])
print('Full legacy inventory matches Run 004. No fitting or backward pass was performed.')

## 3. Xem Run 004 đã học và so baseline

Các biểu đồ dưới đây đọc **600 log training và 15 dòng metric đã được xác minh**, lưu cùng source đã pin. Đây là kết quả Run 004, không phải lượt train mới. Đường đậm là trung bình trượt 20 bước để dễ nhìn; đường nhạt giữ nguyên loss minibatch. Loss này khác đơn vị với MAE lợi suất.

Run 004 **không ghi validation loss tại từng bước**, vì vậy chưa thể xác định lúc bắt đầu overfit. Baseline return bằng 0 phải xuất hiện cùng Chronos zero-shot và cả ba seed; không chỉ so với baseline yếu hơn. Giới hạn này cũng áp dụng khi training loss giảm.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

VISUALS=WORK/'visual-review'
VISUALS.mkdir(exist_ok=True)
review=CODE/'reports/experiments/run-004-seed-stability'
loss=pd.read_csv(review/'training-loss-verified.csv')
historical=pd.read_csv(review/'metrics-verified.csv')
if len(loss)!=600 or set(loss.seed)!={42,123,2026}:
    raise ValueError('Run 004 verified training records differ')
fig,axes=plt.subplots(2,2,figsize=(14,9),layout='constrained')
for seed,part in loss.groupby('seed',sort=True):
    part=part.sort_values('step')
    line=axes[0,0].plot(part.step,part.loss,alpha=.15)[0]
    axes[0,0].plot(part.step,part.loss.rolling(20,min_periods=1).mean(),
                   color=line.get_color(),label=f'Seed {seed}: rolling 20')
    axes[0,1].plot(part.step,part.learning_rate,label=f'Seed {seed}')
    axes[1,0].plot(part.step,part.grad_norm,label=f'Seed {seed}',alpha=.7)
axes[0,0].set(title='Run 004 training loss: minibatch / rolling mean',xlabel='Optimizer step',ylabel='Training objective')
axes[0,1].set(title='Recorded learning-rate schedule',xlabel='Optimizer step',ylabel='Learning rate')
axes[1,0].set(title='Recorded gradient norm',xlabel='Optimizer step',ylabel='Gradient norm')
pooled=historical[historical.asset.eq('ALL')].copy()
names=[f'LoRA seed {int(row.seed)}' if row.model=='lora_B' else row.model for row in pooled.itertuples()]
axes[1,1].barh(names,pooled.mae*1e4,color=['#64748b','#94a3b8','#2563eb','#0891b2','#0d9488'])
axes[1,1].invert_yaxis()
axes[1,1].set(title='Same 512 development windows; lower MAE is better',xlabel='MAE (basis points of return)')
for ax in axes.flat:
    ax.grid(alpha=.2)
for ax in (axes[0,0],axes[0,1],axes[1,0]):
    ax.legend(fontsize=8)
fig.suptitle('Run 004 evidence — no step-wise validation curve was recorded',fontsize=14)
fig.savefig(VISUALS/'run004-training-and-baselines.png',dpi=150)
plt.show()
plt.close(fig)
display(historical[['asset','model','seed','rows','mae','mse','directional_accuracy']])
settings={key:old_lock[key] for key in ('model_id','revision','steps','seeds','context_length','batch_size','learning_rate')}
settings.update({key:old_lock['lora'][key] for key in ('r','lora_alpha','lora_dropout','target_modules')})
display(pd.DataFrame({'setting':list(settings),'recorded_value':[str(value) for value in settings.values()]}))

## 4. Chạy đối chứng và tải ZIP

Runner dùng cùng checkpoint, adapter, batch 16, context 128, horizon và cách chuyển forecast log-price thành return của Run 004. Adapter được nạp không merge; tất cả parameters bị freeze. Mỗi lượt ghi mode, RNG fingerprints, full model-state hash trước/sau, dự báo từng hàng và metric theo tài sản.

Các case đã đặt trước: eval cùng RNG hai lần; eval khác RNG; dropout bật cùng RNG hai lần; dropout bật với RNG 123/2026; cuối cùng eval lại sau các lượt dropout. Bản đối chiếu cùng seed 42 của Run 004 cũng được giữ trong ZIP. Không chọn lượt có MAE đẹp nhất.

Nếu các control hoặc reference không khớp tolerance rtol 1e-6 / atol 1e-9, báo cáo ghi rõ; không tự sửa precision hoặc chạy lại. Nếu trọng số/buffer thay đổi hoặc có NaN, runner dừng và giữ partial evidence.

In [ ]:
from quantstream.inference_lab import prepare_inputs,run_inference_lab
from quantstream.seed_lab import archive_evidence
try:
    snapshot,reference=prepare_inputs(Path('/kaggle/input'),WORK)
    report=run_inference_lab(snapshot,reference,OUTPUT,device='cuda')
    print('Optimizer steps:',report['optimizer_steps'])
    print('Controls passed:',report['controls_passed'])
    print('Run 004 reference reproduced:',report['reference_reproduced'])
    print('Đã xong inference diagnostic. Tải QuantStream_Run005_Retry001_Output.zip và gửi lại.')
except BaseException:
    if not OUTPUT.exists():
        OUTPUT.mkdir(parents=True)
        (OUTPUT/'input-error.txt').write_text(traceback.format_exc(),encoding='utf-8')
        (OUTPUT/'status.json').write_text(json.dumps({'status':'input_failed','optimizer_steps':0}),encoding='utf-8')
    raise
finally:
    if OUTPUT.exists() and SETUP.exists():
        shutil.copytree(SETUP,OUTPUT/'setup',dirs_exist_ok=True)
    saved=archive_evidence(OUTPUT,ARCHIVE)
    if saved:
        display(FileLink(saved))

## 5. Xem từng lượt inference, cấu hình model và control

Chỉ vẽ từ output đã chạy hoàn tất. So **tất cả tám lượt**, cùng zero-return và zero-shot trên đúng các dòng đó. Biểu đồ scatter giữ vị trí thời gian thật của mẫu thưa; không nối thành giá thị trường liên tục. Không có validation/training curve mới vì Run 005 không train.

`dense_act_fn`/`feed_forward_proj`, nếu có trong model config thực tế, được hiển thị nguyên trạng. Run này giữ kiến trúc checkpoint cố định, không đổi activation rồi gọi đó là cùng pretrained model. Các hình được thêm vào ZIP sau khi vẽ; nếu cell vẽ lỗi, ZIP evidence của cell trước vẫn còn.

In [ ]:
if report['completed_passes']!=8:
    raise ValueError('Need the completed eight-pass diagnostic before plotting')
reference=pd.read_parquet(OUTPUT/'reference_predictions.parquet')
base=pd.read_parquet(OUTPUT/'base_control.parquet')
passes={row['case']:pd.read_parquet(OUTPUT/'passes'/(row['case']+'.parquet')) for row in report['passes']}
labels=['zero_return','zero_shot',*passes]
fig,axes=plt.subplots(1,3,figsize=(16,6),layout='constrained')
for ax,asset in zip(axes,('ALL','BTCUSDT','ETHUSDT')):
    mask=np.ones(len(reference),dtype=bool) if asset=='ALL' else reference.symbol.eq(asset).to_numpy()
    truth=reference.target_return.to_numpy()[mask]
    values=[np.mean(abs(truth)),np.mean(abs(base.prediction.to_numpy()[mask]-truth))]
    values.extend(np.mean(abs(frame.prediction.to_numpy()[mask]-truth)) for frame in passes.values())
    ax.barh(labels,np.asarray(values)*1e4,color=['#64748b','#94a3b8']+['#2563eb' if name.startswith('eval') else '#d97706' for name in passes])
    ax.invert_yaxis()
    ax.set(title=f'{asset}: {mask.sum()} rows',xlabel='MAE (basis points of return)')
    ax.grid(axis='x',alpha=.2)
fig.suptitle('Run 005: fixed adapter, every predeclared pass — lower error is better')
fig.savefig(VISUALS/'run005-baselines-and-passes.png',dpi=150)
plt.show()
plt.close(fig)
fig,axes=plt.subplots(2,1,figsize=(14,8),layout='constrained')
for ax,asset in zip(axes,('BTCUSDT','ETHUSDT')):
    mask=reference.symbol.eq(asset).to_numpy()
    times=pd.to_datetime(reference.loc[mask,'decision_time'],utc=True)
    ax.scatter(times,reference.target_return.to_numpy()[mask]*1e4,s=10,alpha=.5,label='Actual target return')
    for case,color in (('eval_42_a','#2563eb'),('dropout_42_a','#d97706')):
        ax.scatter(times,passes[case].prediction.to_numpy()[mask]*1e4,s=10,alpha=.6,label=case,color=color)
    ax.axhline(0,color='#64748b',linewidth=.7)
    ax.set(title=f'{asset}: the same sampled development windows',xlabel='Decision time (UTC)',ylabel='Return (basis points)')
    ax.legend(fontsize=8)
    ax.grid(alpha=.2)
fig.savefig(VISUALS/'run005-actual-and-predicted-returns.png',dpi=150)
plt.show()
plt.close(fig)
checks=pd.DataFrame(report['checks']).T
display(checks)
display(pd.DataFrame(report['passes'])[['case','rng_seed','model_training_before','model_training_after',
    'active_dropout_modules','merged_lora_layers','state_unchanged','trainable_parameters','optimizer_steps']])
config=json.loads((OUTPUT/'base_model_config.json').read_text())
keys=('model_type','architectures','d_model','d_ff','d_kv','num_layers','num_heads','dropout_rate','dense_act_fn','feed_forward_proj')
display(pd.DataFrame({'config_key':keys,'actual_value':[str(config.get(key,'not recorded in this config')) for key in keys]}))
import shutil
shutil.copytree(VISUALS,OUTPUT/'figures',dirs_exist_ok=True)
saved=archive_evidence(OUTPUT,ARCHIVE)
display(FileLink(saved))
print('ZIP updated with figures. All model decisions and optimizer_steps=0 remain unchanged.')

## Cách đọc kết quả

`completed` xác nhận đủ lượt đã chạy, không tự xác nhận control/reference đều đạt. Đọc riêng `controls_passed`, `reference_reproduced`, model-state hashes và mọi dự báo. Các RNG seed trong lab là seed của **inference**, không phải training seed mới; adapter luôn là bản seed 42.

Một mode effect trong môi trường này chưa tái dựng được chính xác runtime/RNG lịch sử không được lưu ở Run 003. Dữ liệu development cũ không trở thành test độc lập. Giữ `research_only_no_promotion`; không có PnL, forward evaluation hoặc model được triển khai.